# Validation Suite — Telegraph Model SSA

Three independent checks that the Gillespie SSA (`simulate_telegraph`) together with the
real-time moment estimator (`compute_sample_moments`) reproduce known analytical results
of the two-state (telegraph) gene-expression model.

1. **Linear growth** — with no degradation the mean RNA grows as $\mathbb{E}[R(t)] = k_{syn}\,t$.
2. **Poisson steady state** — a constitutively active gene gives $R \sim \text{Poisson}(k_{syn}/k_{deg})$, with Fano factor $=1$.
3. **Convergence** — the SSA mean approaches the deterministic ODE solution as $n_{rep}$ grows.

Each case prints a clear PASS/FAIL line and shows a supporting figure.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import plotly.graph_objects as go
from scipy import stats

from src.ssa_simulation import simulate_telegraph, compute_sample_moments
from src.ode_moments import solve_ode_moments
from src.ssa_visualization import show_sample_moments, show_single_trajectory

---
## Case 1 — Linear Growth Rate (no degradation, $k_{deg}=0$)

**Setup:** gene locked ON (`k_off = 0`, `g0 = 1`), synthesis `k_syn = 2.5`, no
degradation (`k_deg = 0`).

**Scientific logic:** with the gene always active and nothing removing RNA, synthesis is
a pure Poisson birth process, so the ensemble mean grows strictly linearly:

$$\mathbb{E}[R(t)] = k_{syn}\,t.$$

**Test:** fit a linear regression of $\langle R \rangle$ over the *entire* real-time grid
(`scipy.stats.linregress`) and require $R^2 > 0.99$ with slope $\approx k_{syn}$
(`np.isclose`, atol = 0.05).

In [ ]:
k_syn = 2.5
data = simulate_telegraph(
    k_on=0.1, k_off=0.0, k_syn=k_syn, k_deg=0.0,
    t0=0.0, g0=1, r0=0, n_sim=2000, n_rep=500,
)
moments = compute_sample_moments(data)   # ensemble mean on a real-time grid (ZOH)

# Linear regression of the ensemble-mean RNA over the whole real-time grid.
fit = stats.linregress(moments["time"], moments["mu_R"])
slope, r_squared = fit.slope, fit.rvalue ** 2

slope_ok = np.isclose(slope, k_syn, atol=0.05)
fit_ok = r_squared > 0.99

print(f"Slope     : {slope:.4f}   (expected k_syn = {k_syn})  "
      f"-> {'PASSED' if slope_ok else 'FAILED'}")
print(f"R-squared : {r_squared:.5f}  (threshold > 0.99)        "
      f"-> {'PASSED' if fit_ok else 'FAILED'}")
print(f"\nCASE 1: {'PASSED' if (slope_ok and fit_ok) else 'FAILED'}")
assert slope_ok and fit_ok

In [ ]:
show_sample_moments(moments, title="Case 1 — Linear RNA Growth (k_deg = 0)")
show_single_trajectory(
    data, trajectory_index=0, title="Case 1 — Single Realization (staircase)"
)

---
## Case 2 — Steady-State Poisson Distribution (instant activation, $k_{off}=0$)

**Setup:** gene locked ON (`k_off = 0`, `g0 = 1`), `k_syn = 10.0`, `k_deg = 0.2`.

**Scientific logic:** a constitutively active gene is a simple birth–death process whose
steady state is a Poisson distribution with mean $\mu = k_{syn}/k_{deg} = 50$ and Fano
factor (Variance / Mean) exactly $1.0$.

**Tests:**
1. Fano factor $\approx 1$ over the steady-state window.
2. Histogram of the steady-state RNA counts overlaid with the theoretical Poisson PMF.
3. A chi-square goodness-of-fit test confirming the Poisson distribution ($p > 0.05$).

*Note:* the steady-state samples are taken at a **fixed real time** (via zero-order
hold), not at event indices — sampling at reaction events would bias the distribution
toward higher-propensity states.

In [ ]:
k_syn, k_deg = 10.0, 0.2
mu_theory = k_syn / k_deg          # 50

data = simulate_telegraph(
    k_on=0.1, k_off=0.0, k_syn=k_syn, k_deg=k_deg,
    t0=0.0, g0=1, r0=0, n_sim=3000, n_rep=2000,
)
moments = compute_sample_moments(data)

# Fano factor over the steady-state window (last half of the real-time grid).
tail = moments["time"] >= 0.5 * moments["time"][-1]
mean_R = moments["mu_R"][tail].mean()
fano = (moments["sigma_R"][tail] ** 2).mean() / mean_R
fano_ok = np.isclose(fano, 1.0, atol=0.15)

print(f"Steady-state mean R : {mean_R:.2f}   (expected {mu_theory:.0f})")
print(f"Fano (Var/Mean)     : {fano:.3f}  (expected 1.0)  "
      f"-> {'PASSED' if fano_ok else 'FAILED'}")
assert fano_ok

show_sample_moments(moments, title="Case 2 — Convergence to Poisson Steady State")

In [ ]:
# Time-stationary samples: each trajectory's RNA at one fixed real time (ZOH).
# (Sampling at event indices would bias toward high-propensity states.)
t_sample = data[-1, :, 0].min()          # latest time every trajectory still covers
final_R = np.empty(data.shape[1], dtype=int)
for j in range(data.shape[1]):
    idx = np.searchsorted(data[:, j, 0], t_sample, side="right") - 1
    final_R[j] = data[idx, j, 2]

k = np.arange(final_R.min(), final_R.max() + 1)
pmf = stats.poisson.pmf(k, mu_theory)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=final_R, histnorm="probability",
    xbins=dict(start=final_R.min() - 0.5, end=final_R.max() + 0.5, size=1),
    marker_color="rgba(239, 68, 68, 0.55)", name="SSA samples",
))
fig.add_trace(go.Scatter(
    x=k, y=pmf, mode="lines+markers",
    line=dict(color="#1E40AF", width=2), name=f"Poisson(μ={mu_theory:.0f})",
))
fig.update_layout(
    template="plotly_white", width=800, height=450, bargap=0.05, hovermode="x",
    title="<b>Case 2 — Final RNA Distribution vs Poisson PMF</b>",
    xaxis_title="RNA count R", yaxis_title="Probability",
)
fig.show()

# Chi-square goodness-of-fit vs Poisson(50). Keep bins with expected >= 5 and fold the
# sparse tails into the first/last kept bin (standard chi-square practice).
obs = np.bincount(final_R, minlength=k.max() + 1)[k.min():].astype(float)
exp = final_R.size * stats.poisson.pmf(k, mu_theory)

keep = np.where(exp >= 5)[0]
lo, hi = keep[0], keep[-1]
obs_b = obs[lo:hi + 1].copy(); obs_b[0] += obs[:lo].sum(); obs_b[-1] += obs[hi + 1:].sum()
exp_b = exp[lo:hi + 1].copy(); exp_b[0] += exp[:lo].sum(); exp_b[-1] += exp[hi + 1:].sum()
exp_b *= obs_b.sum() / exp_b.sum()       # match totals exactly for chisquare

chi2, p_value = stats.chisquare(obs_b, exp_b)
poisson_ok = p_value > 0.05

print(f"Chi-square fit : chi2 = {chi2:.2f}, p = {p_value:.3f}  "
      f"-> {'PASSED (consistent with Poisson)' if poisson_ok else 'FAILED'}")
print(f"\nCASE 2: {'PASSED' if (fano_ok and poisson_ok) else 'FAILED'}")
assert poisson_ok

---
## Case 3 — Convergence Test (ensemble-size scaling)

**Scientific logic:** the SSA ensemble mean is an unbiased estimator of the
deterministic ODE moment solution, so as the number of trajectories `n_rep` increases the
empirical mean must converge to the ODE curve, with the error shrinking like
$1/\sqrt{n_{rep}}$.

**Test:** for `n_rep ∈ {10, 100, 1000}`, compare $\langle R \rangle_{SSA}$ against
`solve_ode_moments` on the shared time grid and confirm the mean absolute error
decreases.

In [ ]:
kin = dict(k_on=0.1, k_off=0.1, k_syn=10.0, k_deg=0.2)
n_reps = [10, 100, 1000]
errors = []

for n_rep in n_reps:
    data = simulate_telegraph(**kin, t0=0.0, g0=0, r0=0, n_sim=2000, n_rep=n_rep)
    m = compute_sample_moments(data)                 # grid: linspace(t0, t_end, 1000)
    t_end = float(m["time"][-1])
    _, y = solve_ode_moments(**kin, t0=0.0, g0=0, r0=0, t_end=t_end)  # same grid
    err = np.mean(np.abs(m["mu_R"] - y[1]))          # y[1] = ODE mean R
    errors.append(err)
    print(f"n_rep = {n_rep:5d}  |  mean |SSA - ODE| = {err:.4f}")

converged = errors[-1] < errors[0]
print(f"\nCASE 3: {'PASSED' if converged else 'FAILED'}  "
      f"(error {errors[0]:.4f} @ n=10  ->  {errors[-1]:.4f} @ n=1000)")
assert converged

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=n_reps, y=errors, mode="lines+markers",
    line=dict(color="#10B981", width=2), name="mean |SSA - ODE|",
))
guide = errors[0] * np.sqrt(n_reps[0] / np.array(n_reps, float))   # 1/sqrt(n) reference
fig.add_trace(go.Scatter(
    x=n_reps, y=guide, mode="lines",
    line=dict(color="gray", dash="dot"), name="∝ 1/√n_rep",
))
fig.update_layout(
    template="plotly_white", width=700, height=450,
    title="<b>Case 3 — SSA → ODE Convergence vs Ensemble Size</b>",
    xaxis_title="n_rep", yaxis_title="mean abs error",
)
fig.update_xaxes(type="log")
fig.update_yaxes(type="log")
fig.show()